# MSE 446 Project Notebook

This notebook puts the current repo source into one document. The original code and comments are preserved as closely as possible, with markdown section breakes to make the notebook easier to navigate.

Included Stuff:
- README / repo notes
- Raw data `Traffic_Collisions_280340447332117481.csv`
- `kw_heatmap/pipeline.py`
- `build_kw_heatmap.py`
- `negative_sampling.py`
- `data_processing.py`
- `app.py`
- `train_model.py`
- `train_multiple_models.py`

Not included:
- generated artifacts from `artifacts/`
- cached GIS data from `cache/`
- local environment folders like `sklearn-env/`


## Suggested Reading / Execution Order

1. Read the README section.
2. Review the pipeline code that builds the heatmap artifacts.
3. Review the negative-sampling and data-processing scripts.
4. Review the model training scripts.
5. Review the Streamlit app code last, since it depends on the built artifacts.

Some code cells are copied directly from repo scripts and may execute immediately if you run them as-is in Jupyter, especially the script-style files with `if __name__ == "__main__":` blocks.


## README

Source: `README.md`


# KW Collision Zone Heatmap

## Components

- `build_kw_heatmap.py` / `kw_heatmap/pipeline.py`: zone boundary construction and collision assignment logic
- `app.py`: Streamlit heatmap visualization
- `data_processing.py`: cleans the raw collision CSV and applies one-hot encoding to categorical features
- `negative_sampling.py`: generates synthetic non-collision records and combines them with the real collision data for model training
- `train_model.py`: trains and evaluates Random Forest and Gradient Boosting classifiers
- `train_multiple_models.py`: trains and evaluates multiple basic and adanced classfiers.

---

## Full pipeline — run in this order

### 1. Install dependencies

```bash
python -m pip install -r requirements.txt
```

### 2. Build zone map and assign collisions to zones

```bash
python build_kw_heatmap.py
```

Optional flags:

```bash
python build_kw_heatmap.py --target-zones 50 --refresh-cache
```

Outputs to `artifacts/`:

- `collisions_cleaned.csv` — collision records with zone assignments
- `collisions_unmappable.csv` — records excluded due to missing/zero coordinates
- `zones.geojson` — zone boundary polygons
- `zone_metrics.csv` — per-zone collision counts and density
- `build_summary.json` — build validation summary

GIS inputs are cached in `cache/`. The first build downloads and polygonizes the OSM street network for Kitchener-Waterloo and may take a few minutes. Later builds reuse the cache.
NOTE: This command can take a while to complete.

### 3. Generate negative samples

```bash
python negative_sampling.py
```

Reads `artifacts/collisions_cleaned.csv` (produced in step 2) and generates synthetic non-collision records in the same raw format as the real data. Negatives are allocated proportionally per zone — zones with more real collisions receive more negatives, reflecting higher traffic exposure in those areas. Each negative row is assigned a plausible combination of categorical features (light condition, road type, traffic control, etc.) sampled from the empirical distribution of real collisions in that zone.

Adds a `CRASH` column to both real (`1`) and synthetic (`0`) rows, then saves the combined dataset.

Output:

- `artifacts/collisions_with_negatives.csv` — balanced dataset ready for preprocessing

### 4. Preprocess and encode features

```bash
python data_processing.py
```

Reads `artifacts/collisions_with_negatives.csv`, drops unnecessary columns, applies one-hot encoding to all categorical features, and saves the processed result.

> **Note:** make sure `data_processing.py` reads from `artifacts/collisions_with_negatives.csv` and that `CRASH` is not in the `columns_to_del` list so it passes through as the label column.

Output:

- `Traffic_Collisions_Updated.csv` — fully processed and encoded dataset

### 5. Train the model

```bash
python train_model.py
```

Loads `Traffic_Collisions_Updated.csv` and runs an 80/20 stratified train/test split. Encodes `ACCIDENTDATE` as a numeric ordinal and one-hot encodes `zone_id` so both are usable as features.
Shows statistics and ROC-AUC Curve for best model - Gradient Boosting.

```bash
Optional command: python train_multiple_models.py
```
Shows various models we tested before finding best model.

**Baseline models** (with `RandomizedSearchCV` tuning):
- `LogisticRegression` — tunes `C` and `l1_ratio`
- `DecisionTree` — tunes `max_depth`, `criterion`, and `min_samples_leaf`

**Advanced models** (with `RandomizedSearchCV` tuning):
- `RandomForest` — tunes `n_estimators`, `max_depth`, `max_features`, `min_samples_leaf`
- `GradientBoosting` — tunes `n_estimators`, `learning_rate`, `max_depth`, `subsample`
- `HistGradientBoosting` — sklearn's XGBoost-equivalent, fast and accurate

For each model: prints accuracy, ROC-AUC, 5-fold CV score (mean ± std), classification report, and confusion matrix. Also prints top 15 feature importances from Random Forest.

Saves the best model (by ROC-AUC) to `artifacts/`:

- `collision_model.pkl` — trained model
- `model_features.pkl` — feature list
- `model_summary.json` — evaluation summary

### 6. Launch the heatmap app

```bash
streamlit run app.py
```

Opens automatically at `http://localhost:8501`.

---

## Notes

- The heatmap uses the raw collision CSV as the source of truth because the processed CSV drops latitude and longitude.
- Collisions with zero or missing coordinates are excluded from the map and reported in the UI.
- Zone boundaries are derived from public drivable OSM streets clipped to the municipal boundaries of Kitchener and Waterloo.
- Negative sampling falls back to global (non-zone-proportional) sampling if `collisions_cleaned.csv` is not yet available. Re-run `negative_sampling.py` after step 2 to get the full zone-proportional behavior.

## AI Usage

Claude Opus 4.6 and GPT-5.4 were used during development to help with debugging, code suggestions and understanding concepts. All final decisions and implementation were made by the members.


## Pipeline

Source: `kw_heatmap/pipeline.py`


In [ ]:
from __future__ import annotations

import argparse
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import geopandas as gpd
import networkx as nx
import osmnx as ox
import pandas as pd
from shapely.geometry import LineString, MultiLineString
from shapely.ops import polygonize, unary_union

DATE_FORMAT = "%m/%d/%Y %I:%M:%S %p"
BOUNDARY_PLACES = [
    "Kitchener, Ontario, Canada",
    "Waterloo, Ontario, Canada",
]
OUTPUT_CRS = "EPSG:4326"
AREA_CRS = "EPSG:3347"
EXCLUDED_HIGHWAY_TAGS = {
    "service",
    "parking_aisle",
    "driveway",
    "private",
    "emergency_access",
    "rest_area",
    "services",
}
MIN_SHARED_EDGE_M = 1.0
MIN_BLOCK_AREA_M2 = 250.0
NEAREST_ZONE_MAX_DISTANCE_M = 150.0


# Centralized output paths let the builder keep all generated files in
# predictable locations under `artifacts/` and `cache/`.
@dataclass(frozen=True)
class BuildPaths:
    raw_csv: Path
    output_dir: Path
    cache_dir: Path
    cleaned_csv: Path
    unmappable_csv: Path
    zones_geojson: Path
    metrics_csv: Path
    summary_json: Path
    boundary_geojson: Path
    streets_geojson: Path

    @classmethod
    def create(cls, raw_csv: Path, output_dir: Path, cache_dir: Path) -> "BuildPaths":
        return cls(
            raw_csv=raw_csv,
            output_dir=output_dir,
            cache_dir=cache_dir,
            cleaned_csv=output_dir / "collisions_cleaned.csv",
            unmappable_csv=output_dir / "collisions_unmappable.csv",
            zones_geojson=output_dir / "zones.geojson",
            metrics_csv=output_dir / "zone_metrics.csv",
            summary_json=output_dir / "build_summary.json",
            boundary_geojson=cache_dir / "kw_boundary.geojson",
            streets_geojson=cache_dir / "kw_streets.geojson",
        )


def ensure_directory(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


# GeoJSON is used as the interchange format for generated geography so the
# Streamlit app can read the outputs without depending on shapefile sidecars
def write_geojson(gdf: gpd.GeoDataFrame, path: Path) -> None:
    ensure_directory(path.parent)
    payload = json.loads(gdf.to_crs(OUTPUT_CRS).to_json(drop_id=True))
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def read_geojson(path: Path, crs: str = OUTPUT_CRS) -> gpd.GeoDataFrame:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    features = payload.get("features", [])
    if not features:
        return gpd.GeoDataFrame(geometry=[], crs=crs)
    return gpd.GeoDataFrame.from_features(features, crs=crs)


def normalize_highway_tags(value: object) -> set[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return set()
    if isinstance(value, (list, tuple, set)):
        tags = value
    else:
        tags = [value]
    return {str(tag).strip().lower() for tag in tags if str(tag).strip()}


def is_public_street(value: object) -> bool:
    tags = normalize_highway_tags(value)
    if not tags:
        return True
    return tags.isdisjoint(EXCLUDED_HIGHWAY_TAGS)


def iter_lines(geometry) -> Iterable[LineString]:
    if geometry is None or geometry.is_empty:
        return
    if isinstance(geometry, LineString):
        yield geometry
    elif isinstance(geometry, MultiLineString):
        for part in geometry.geoms:
            if not part.is_empty:
                yield part


def clean_collision_data(raw_csv: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    # Separate valid point records from rows that cannot be mapped, while also
    # extracting the small set of summary fields used later in validation/UI.
    df = pd.read_csv(raw_csv)
    df["source_row"] = df.index
    df["ACCIDENTDATE"] = pd.to_datetime(df["ACCIDENTDATE"], format=DATE_FORMAT)
    df["year"] = df["ACCIDENTDATE"].dt.year
    df["has_valid_coords"] = (
        df["LATITUDE"].notna()
        & df["LONGITUDE"].notna()
        & (df["LATITUDE"] != 0)
        & (df["LONGITUDE"] != 0)
    )

    valid = df[df["has_valid_coords"]].copy()
    invalid = df[~df["has_valid_coords"]].copy()
    summary = {
        "raw_rows": int(len(df)),
        "mappable_rows": int(len(valid)),
        "unmappable_rows": int(len(invalid)),
        "date_min": valid["ACCIDENTDATE"].min().isoformat(),
        "date_max": valid["ACCIDENTDATE"].max().isoformat(),
    }
    return valid, invalid, summary


def fetch_kw_boundary(cache_path: Path, refresh_cache: bool = False) -> gpd.GeoDataFrame:
    if cache_path.exists() and not refresh_cache:
        return read_geojson(cache_path)

    # The study area is defined as the union of Kitchener and Waterloo so the
    # zoning step works from one shared municipal footprint.
    geocoded_frames = []
    for place in BOUNDARY_PLACES:
        place_gdf = ox.geocode_to_gdf(place)
        geocoded_frames.append(place_gdf[["display_name", "geometry"]].copy())

    boundary = gpd.GeoDataFrame(
        pd.concat(geocoded_frames, ignore_index=True),
        geometry="geometry",
        crs=geocoded_frames[0].crs,
    )
    dissolved = gpd.GeoDataFrame(
        {"name": ["Kitchener-Waterloo"], "geometry": [unary_union(boundary.geometry)]},
        crs=boundary.crs,
    )
    write_geojson(dissolved, cache_path)
    return dissolved


def fetch_kw_streets(
    boundary: gpd.GeoDataFrame,
    cache_path: Path,
    refresh_cache: bool = False,
) -> gpd.GeoDataFrame:
    if cache_path.exists() and not refresh_cache:
        return read_geojson(cache_path)

    # Download the drivable street network inside the study boundary, then clip
    # and cache it so later runs do not need to hit OSM again.
    ox.settings.use_cache = True
    ox.settings.log_console = False
    graph = ox.graph_from_polygon(
        boundary.to_crs(OUTPUT_CRS).geometry.iloc[0],
        network_type="drive",
        simplify=True,
        retain_all=True,
        truncate_by_edge=True,
    )
    edges = ox.graph_to_gdfs(graph, nodes=False, edges=True).reset_index(drop=True)
    edges = edges[edges["highway"].apply(is_public_street)].copy()
    edges = gpd.clip(edges, boundary.to_crs(edges.crs))
    keep_cols = [column for column in ["name", "highway", "geometry"] if column in edges.columns]
    streets = edges[keep_cols].explode(index_parts=False, ignore_index=True)
    streets = streets[streets.geometry.notna() & ~streets.geometry.is_empty].copy()
    write_geojson(streets, cache_path)
    return streets


def build_street_blocks(
    boundary: gpd.GeoDataFrame,
    streets: gpd.GeoDataFrame,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    # Polygonizing the street lines turns the road network into base spatial
    # blocks that can later be merged into larger analysis zones.
    boundary_proj = boundary.to_crs(AREA_CRS)
    streets_proj = streets.to_crs(AREA_CRS)
    study_area = boundary_proj.geometry.iloc[0]

    line_geometries = []
    for geometry in streets_proj.geometry:
        clipped = geometry.intersection(study_area)
        for line in iter_lines(clipped):
            line_geometries.append(line)
    for line in iter_lines(study_area.boundary):
        line_geometries.append(line)

    polygon_geometries = list(polygonize(unary_union(line_geometries)))
    blocks = gpd.GeoDataFrame(geometry=polygon_geometries, crs=AREA_CRS)
    blocks = blocks[blocks.geometry.notna() & ~blocks.geometry.is_empty].copy()
    blocks["geometry"] = blocks.geometry.intersection(study_area)
    blocks = blocks[blocks.geometry.notna() & ~blocks.geometry.is_empty].copy()
    blocks = blocks[blocks.area > MIN_BLOCK_AREA_M2].copy()
    blocks["inside"] = blocks.representative_point().within(study_area.buffer(1e-6))
    blocks = blocks[blocks["inside"]].drop(columns=["inside"]).copy()
    blocks = blocks.explode(index_parts=False, ignore_index=True)
    blocks["block_id"] = range(len(blocks))
    blocks["area_m2"] = blocks.area
    blocks["centroid"] = blocks.centroid
    return blocks, boundary_proj


def build_block_graph(blocks: gpd.GeoDataFrame) -> nx.Graph:
    # Adjacency is tracked as a graph so zone growth/rebalancing can preserve
    # contiguity while still operating on street-bounded block geometry.
    graph = nx.Graph()
    blocks = blocks.reset_index(drop=True)
    for row in blocks.itertuples():
        graph.add_node(int(row.block_id), area=float(row.area_m2))

    sindex = blocks.sindex
    for row in blocks.itertuples():
        candidate_indexes = list(sindex.query(row.geometry))
        for candidate_index in candidate_indexes:
            other = blocks.iloc[candidate_index]
            other_id = int(other["block_id"])
            if other_id <= row.block_id:
                continue
            if not row.geometry.touches(other.geometry):
                continue
            shared_length = row.geometry.boundary.intersection(other.geometry.boundary).length
            if shared_length <= MIN_SHARED_EDGE_M:
                continue
            graph.add_edge(int(row.block_id), other_id, shared_length=float(shared_length))
    return graph


def allocate_seed_counts(
    graph: nx.Graph,
    block_areas: dict[int, float],
    target_zones: int,
) -> list[tuple[list[int], int]]:
    # Split the requested zone count across connected components so isolated
    # pieces of geometry still receive at least one zone.
    components = [sorted(component) for component in nx.connected_components(graph)]
    target_zones = max(target_zones, len(components))
    target_zones = min(target_zones, graph.number_of_nodes())

    component_areas = [sum(block_areas[node] for node in component) for component in components]
    total_area = sum(component_areas)
    seed_counts = [1] * len(components)

    remaining = target_zones - len(components)
    if remaining > 0:
        quotas = [
            (area / total_area) * remaining if total_area else 0.0
            for area in component_areas
        ]
        extras = []
        for component, quota in zip(components, quotas):
            cap = max(len(component) - 1, 0)
            extras.append(min(cap, math.floor(quota)))
        seed_counts = [base + extra for base, extra in zip(seed_counts, extras)]
        assigned = sum(seed_counts)

        remainders = [quota - math.floor(quota) for quota in quotas]
        while assigned < target_zones:
            candidates = [
                index
                for index, component in enumerate(components)
                if seed_counts[index] < len(component)
            ]
            if not candidates:
                break
            chosen = max(
                candidates,
                key=lambda index: (remainders[index], component_areas[index], -index),
            )
            seed_counts[chosen] += 1
            assigned += 1
            remainders[chosen] = 0.0

    return list(zip(components, seed_counts))


def choose_seed_nodes(
    blocks: gpd.GeoDataFrame,
    component_nodes: list[int],
    seed_count: int,
) -> list[int]:
    # Farthest-point style seeding spreads the initial zone anchors across each
    # component before the contiguous growth phase begins.
    block_lookup = blocks.set_index("block_id")
    centroids = {node: block_lookup.at[node, "centroid"] for node in component_nodes}
    areas = {node: float(block_lookup.at[node, "area_m2"]) for node in component_nodes}
    component_centroid = unary_union([centroids[node] for node in component_nodes]).centroid

    first_seed = min(
        component_nodes,
        key=lambda node: (
            centroids[node].distance(component_centroid),
            -areas[node],
            centroids[node].x,
            centroids[node].y,
            node,
        ),
    )
    seeds = [first_seed]
    remaining = [node for node in component_nodes if node != first_seed]

    while len(seeds) < seed_count and remaining:
        next_seed = max(
            remaining,
            key=lambda node: (
                min(centroids[node].distance(centroids[seed]) for seed in seeds),
                areas[node],
                -centroids[node].x,
                -centroids[node].y,
                -node,
            ),
        )
        seeds.append(next_seed)
        remaining.remove(next_seed)
    return seeds


def shared_boundary_to_zone(
    graph: nx.Graph,
    assignments: dict[int, int],
    node: int,
    zone_id: int,
) -> float:
    shared = 0.0
    for neighbor in graph.neighbors(node):
        if assignments.get(neighbor) == zone_id:
            shared += float(graph.edges[node, neighbor].get("shared_length", 0.0))
    return shared


def grow_component_zones(
    graph: nx.Graph,
    blocks: gpd.GeoDataFrame,
    component_nodes: list[int],
    seed_nodes: list[int],
    zone_ids: list[int],
) -> dict[int, int]:
    # Grow each zone by taking adjacent blocks until the component is fully
    # assigned, which keeps every intermediate zone contiguous.
    block_lookup = blocks.set_index("block_id")
    area_lookup = block_lookup["area_m2"].to_dict()
    centroid_lookup = block_lookup["centroid"].to_dict()
    component_target_area = sum(area_lookup[node] for node in component_nodes) / max(len(zone_ids), 1)
    component_set = set(component_nodes)

    assignments = {seed: zone_id for seed, zone_id in zip(seed_nodes, zone_ids)}
    zone_areas = {zone_id: float(area_lookup[seed]) for seed, zone_id in zip(seed_nodes, zone_ids)}
    zone_frontiers = {
        zone_id: {
            neighbor
            for neighbor in graph.neighbors(seed)
            if neighbor in component_set and neighbor not in assignments
        }
        for seed, zone_id in zip(seed_nodes, zone_ids)
    }
    seed_centroids = {zone_id: centroid_lookup[seed] for seed, zone_id in zip(seed_nodes, zone_ids)}
    unassigned = component_set - set(seed_nodes)

    while unassigned:
        active_zones = [zone_id for zone_id, frontier in zone_frontiers.items() if frontier]
        if not active_zones:
            fallback_node = min(unassigned)
            fallback_zone = min(
                zone_ids,
                key=lambda zone_id: (
                    centroid_lookup[fallback_node].distance(seed_centroids[zone_id]),
                    zone_areas[zone_id],
                    zone_id,
                ),
            )
            assignments[fallback_node] = fallback_zone
            zone_areas[fallback_zone] += float(area_lookup[fallback_node])
            unassigned.remove(fallback_node)
            continue

        current_zone = min(active_zones, key=lambda zone_id: (zone_areas[zone_id], zone_id))
        candidate = min(
            zone_frontiers[current_zone],
            key=lambda node: (
                abs((zone_areas[current_zone] + area_lookup[node]) - component_target_area),
                -shared_boundary_to_zone(graph, assignments, node, current_zone),
                centroid_lookup[node].distance(seed_centroids[current_zone]),
                area_lookup[node],
                node,
            ),
        )

        assignments[candidate] = current_zone
        zone_areas[current_zone] += float(area_lookup[candidate])
        unassigned.remove(candidate)
        for zone_id in zone_frontiers:
            zone_frontiers[zone_id].discard(candidate)
        for neighbor in graph.neighbors(candidate):
            if neighbor in unassigned:
                zone_frontiers[current_zone].add(neighbor)
        zone_frontiers[current_zone].discard(candidate)

    return assignments


def source_zone_stays_connected(
    graph: nx.Graph,
    assignments: dict[int, int],
    node_to_move: int,
    donor_zone: int,
) -> bool:
    donor_nodes = [node for node, zone_id in assignments.items() if zone_id == donor_zone and node != node_to_move]
    if not donor_nodes:
        return False
    if len(donor_nodes) == 1:
        return True
    return nx.is_connected(graph.subgraph(donor_nodes))


def rebalance_assignments(
    graph: nx.Graph,
    blocks: gpd.GeoDataFrame,
    assignments: dict[int, int],
    zone_ids: list[int],
    max_passes: int = 200,
) -> dict[int, int]:
    # After the initial growth pass, boundary blocks can be moved between zones
    # if the move improves area balance without breaking contiguity.
    area_lookup = blocks.set_index("block_id")["area_m2"].to_dict()
    zone_areas = {
        zone_id: sum(area_lookup[node] for node, assigned_zone in assignments.items() if assigned_zone == zone_id)
        for zone_id in zone_ids
    }
    target_area = sum(zone_areas.values()) / max(len(zone_ids), 1)

    for _ in range(max_passes):
        best_move = None
        for donor_zone in zone_ids:
            donor_nodes = [node for node, zone_id in assignments.items() if zone_id == donor_zone]
            if len(donor_nodes) <= 1:
                continue

            for node in donor_nodes:
                adjacent_zones = {assignments.get(neighbor) for neighbor in graph.neighbors(node)}
                adjacent_zones.discard(donor_zone)
                for recipient_zone in sorted(zone for zone in adjacent_zones if zone is not None):
                    if not source_zone_stays_connected(graph, assignments, node, donor_zone):
                        continue
                    donor_after = zone_areas[donor_zone] - area_lookup[node]
                    recipient_after = zone_areas[recipient_zone] + area_lookup[node]
                    improvement = (
                        abs(zone_areas[donor_zone] - target_area)
                        + abs(zone_areas[recipient_zone] - target_area)
                        - abs(donor_after - target_area)
                        - abs(recipient_after - target_area)
                    )
                    if improvement <= 1e-9:
                        continue

                    candidate = (
                        improvement,
                        shared_boundary_to_zone(graph, assignments, node, recipient_zone),
                        -abs(recipient_after - target_area),
                        -area_lookup[node],
                        recipient_zone,
                        donor_zone,
                        node,
                    )
                    if best_move is None or candidate > best_move:
                        best_move = candidate

        if best_move is None:
            break

        _, _, _, _, recipient_zone, donor_zone, node = best_move
        assignments[node] = recipient_zone
        zone_areas[donor_zone] -= float(area_lookup[node])
        zone_areas[recipient_zone] += float(area_lookup[node])

    return assignments


def build_analysis_zones(
    blocks: gpd.GeoDataFrame,
    target_zones: int,
) -> tuple[gpd.GeoDataFrame, dict[int, str], nx.Graph]:
    # This is the full zoning stage: build adjacency, place seeds, grow zones,
    # rebalance them, then assign stable human-readable zone IDs.
    graph = build_block_graph(blocks)
    block_areas = blocks.set_index("block_id")["area_m2"].to_dict()
    component_seed_counts = allocate_seed_counts(graph, block_areas, target_zones)

    next_zone_numeric = 0
    assignments: dict[int, int] = {}
    zone_numeric_ids: list[int] = []
    for component_nodes, seed_count in component_seed_counts:
        seeds = choose_seed_nodes(blocks, component_nodes, seed_count)
        zone_ids = list(range(next_zone_numeric, next_zone_numeric + len(seeds)))
        next_zone_numeric += len(seeds)
        zone_numeric_ids.extend(zone_ids)
        assignments.update(grow_component_zones(graph, blocks, component_nodes, seeds, zone_ids))

    assignments = rebalance_assignments(graph, blocks, assignments, zone_numeric_ids)
    blocks_with_zones = blocks.drop(columns=["centroid"]).copy()
    blocks_with_zones["zone_numeric"] = blocks_with_zones["block_id"].map(assignments)

    zones = (
        blocks_with_zones[["zone_numeric", "area_m2", "geometry"]]
        .dissolve(by="zone_numeric", aggfunc={"area_m2": "sum"})
        .reset_index()
    )
    zones["geometry"] = zones.geometry.buffer(0)
    zones["zone_area_km2"] = zones.geometry.area / 1_000_000.0
    centroids = zones.representative_point()
    zones["_centroid_x"] = centroids.x
    zones["_centroid_y"] = centroids.y
    zones = zones.sort_values(by=["_centroid_y", "_centroid_x"], ascending=[False, True]).reset_index(drop=True)
    width = max(2, len(str(len(zones))))
    zones["zone_id"] = [f"Z{index:0{width}d}" for index in range(1, len(zones) + 1)]
    numeric_to_zone_id = dict(zip(zones["zone_numeric"], zones["zone_id"]))
    final_assignments = {block_id: numeric_to_zone_id[zone_numeric] for block_id, zone_numeric in assignments.items()}
    zones = zones.drop(columns=["_centroid_x", "_centroid_y"])
    return zones, final_assignments, graph


def assign_collisions_to_zones(
    collisions: pd.DataFrame,
    zones: gpd.GeoDataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    # Collisions are first joined directly to containing polygons; any boundary
    # edge cases are handled with a nearest-zone fallback.
    points = gpd.GeoDataFrame(
        collisions.copy(),
        geometry=gpd.points_from_xy(collisions["LONGITUDE"], collisions["LATITUDE"]),
        crs=OUTPUT_CRS,
    )
    zone_polygons = zones[["zone_id", "zone_area_km2", "geometry"]].to_crs(OUTPUT_CRS)

    joined = gpd.sjoin(points, zone_polygons, how="left", predicate="within")
    unmatched_mask = joined["zone_id"].isna()
    nearest_assigned = 0
    if unmatched_mask.any():
        nearest = gpd.sjoin_nearest(
            points.loc[unmatched_mask].to_crs(AREA_CRS),
            zone_polygons.to_crs(AREA_CRS),
            how="left",
            max_distance=NEAREST_ZONE_MAX_DISTANCE_M,
            distance_col="distance_to_zone_m",
        )
        joined.loc[unmatched_mask, "zone_id"] = nearest["zone_id"].values
        joined.loc[unmatched_mask, "zone_area_km2"] = nearest["zone_area_km2"].values
        joined.loc[unmatched_mask, "distance_to_zone_m"] = nearest["distance_to_zone_m"].values
        nearest_assigned = int(len(nearest))

    if joined["zone_id"].isna().any():
        missing = int(joined["zone_id"].isna().sum())
        raise ValueError(f"{missing} collisions could not be assigned to a zone.")

    joined = pd.DataFrame(joined.drop(columns=["geometry", "index_right"]))
    years_in_scope = max(joined["year"].nunique(), 1)

    metrics = zones[["zone_id", "zone_area_km2"]].copy()
    counts = joined.groupby("zone_id").size().rename("collision_count").reset_index()
    metrics = metrics.merge(counts, on="zone_id", how="left")
    metrics["collision_count"] = metrics["collision_count"].fillna(0).astype(int)
    metrics["collisions_per_km2"] = metrics["collision_count"] / metrics["zone_area_km2"]
    metrics["avg_collisions_per_year"] = metrics["collision_count"] / years_in_scope

    summary = {
        "assigned_collisions": int(len(joined)),
        "nearest_assigned_collisions": nearest_assigned,
        "years_in_scope": int(years_in_scope),
    }
    return joined, metrics, summary


def zone_assignments_are_contiguous(graph: nx.Graph, assignments: dict[int, str]) -> bool:
    for zone_id in sorted(set(assignments.values())):
        members = [node for node, assigned_zone in assignments.items() if assigned_zone == zone_id]
        if len(members) <= 1:
            continue
        if not nx.is_connected(graph.subgraph(members)):
            return False
    return True


def zones_outside_area_band(
    zones: gpd.GeoDataFrame,
    boundary_proj: gpd.GeoDataFrame,
) -> list[str]:
    median_area = zones["zone_area_km2"].median()
    lower_bound = 0.5 * median_area
    upper_bound = 1.5 * median_area
    study_boundary = boundary_proj.geometry.iloc[0].boundary

    violations = []
    for row in zones.itertuples():
        touches_outer_boundary = row.geometry.boundary.intersection(study_boundary).length > MIN_SHARED_EDGE_M
        if lower_bound <= row.zone_area_km2 <= upper_bound:
            continue
        if touches_outer_boundary:
            continue
        violations.append(row.zone_id)
    return violations


def zone_has_meaningful_rebalance_move(
    zone_id: str,
    graph: nx.Graph,
    blocks: gpd.GeoDataFrame,
    assignments: dict[int, str],
    target_area_m2: float,
    improvement_ratio: float = 0.01,
) -> bool:
    area_lookup = blocks.set_index("block_id")["area_m2"].to_dict()
    zone_areas = {
        assigned_zone: sum(area_lookup[node] for node, candidate_zone in assignments.items() if candidate_zone == assigned_zone)
        for assigned_zone in set(assignments.values())
    }
    threshold = target_area_m2 * improvement_ratio
    current_zone_area = zone_areas[zone_id]

    for node, assigned_zone in assignments.items():
        if assigned_zone == zone_id:
            for neighbor in graph.neighbors(node):
                recipient_zone = assignments[neighbor]
                if recipient_zone == zone_id:
                    continue
                if not source_zone_stays_connected(graph, assignments, node, zone_id):
                    continue
                donor_after = current_zone_area - area_lookup[node]
                recipient_after = zone_areas[recipient_zone] + area_lookup[node]
                improvement = (
                    abs(current_zone_area - target_area_m2)
                    + abs(zone_areas[recipient_zone] - target_area_m2)
                    - abs(donor_after - target_area_m2)
                    - abs(recipient_after - target_area_m2)
                )
                if improvement > threshold:
                    return True
        else:
            if not any(assignments[neighbor] == zone_id for neighbor in graph.neighbors(node)):
                continue
            if not source_zone_stays_connected(graph, assignments, node, assigned_zone):
                continue
            zone_after = current_zone_area + area_lookup[node]
            donor_after = zone_areas[assigned_zone] - area_lookup[node]
            improvement = (
                abs(current_zone_area - target_area_m2)
                + abs(zone_areas[assigned_zone] - target_area_m2)
                - abs(zone_after - target_area_m2)
                - abs(donor_after - target_area_m2)
            )
            if improvement > threshold:
                return True

    return False


def classify_area_band_violations(
    zones: gpd.GeoDataFrame,
    boundary_proj: gpd.GeoDataFrame,
    blocks: gpd.GeoDataFrame,
    assignments: dict[int, str],
    graph: nx.Graph,
) -> tuple[list[str], list[str]]:
    # Area outliers are split into "geometry constrained" vs. "unconstrained"
    # so the validator can distinguish expected edge cases from real failures.
    median_area = zones["zone_area_km2"].median()
    lower_bound = 0.5 * median_area
    upper_bound = 1.5 * median_area
    study_boundary = boundary_proj.geometry.iloc[0].boundary
    target_area_m2 = blocks["area_m2"].sum() / max(len(zones), 1)

    constrained = []
    unconstrained = []
    for row in zones.itertuples():
        if lower_bound <= row.zone_area_km2 <= upper_bound:
            continue
        touches_outer_boundary = row.geometry.boundary.intersection(study_boundary).length > MIN_SHARED_EDGE_M
        if touches_outer_boundary:
            constrained.append(row.zone_id)
            continue
        if not zone_has_meaningful_rebalance_move(
            zone_id=row.zone_id,
            graph=graph,
            blocks=blocks,
            assignments=assignments,
            target_area_m2=target_area_m2,
        ):
            constrained.append(row.zone_id)
            continue
        unconstrained.append(row.zone_id)
    return constrained, unconstrained


def validate_outputs(
    raw_summary: dict,
    cleaned_collisions: pd.DataFrame,
    blocks: gpd.GeoDataFrame,
    zones: gpd.GeoDataFrame,
    metrics: pd.DataFrame,
    assignments: dict[int, str],
    graph: nx.Graph,
    boundary_proj: gpd.GeoDataFrame,
) -> dict:
    # Validation checks are the contract for a successful build: row counts,
    # spatial joins, contiguity, and zone-balance sanity checks.
    validations = {
        "raw_row_count_matches": raw_summary["raw_rows"] == 8928,
        "mappable_row_count_matches": raw_summary["mappable_rows"] == 8598,
        "unmappable_row_count_matches": raw_summary["unmappable_rows"] == 330,
        "date_min_matches": raw_summary["date_min"] == "2015-01-01T00:00:00",
        "date_max_matches": raw_summary["date_max"] == "2022-07-31T17:41:00",
        "zone_count_in_expected_range": 40 <= len(zones) <= 60,
        "all_mappable_collisions_assigned": len(cleaned_collisions) == raw_summary["mappable_rows"],
        "metrics_cover_all_zones": set(metrics["zone_id"]) == set(zones["zone_id"]),
        "zone_graph_contiguous": zone_assignments_are_contiguous(graph, assignments),
    }

    constrained_violations, unconstrained_violations = classify_area_band_violations(
        zones=zones,
        boundary_proj=boundary_proj,
        blocks=blocks,
        assignments=assignments,
        graph=graph,
    )
    validations["interior_zones_within_area_band_or_geometry_constrained"] = not unconstrained_violations
    failed_checks = [name for name, passed in validations.items() if not passed]
    if failed_checks:
        details = ", ".join(failed_checks)
        if unconstrained_violations:
            details += f". Unconstrained zone band violations: {', '.join(unconstrained_violations)}"
        raise ValueError(f"Validation failed for: {details}")
    if constrained_violations:
        validations["geometry_constrained_area_band_warnings"] = constrained_violations
    return validations


def run_pipeline(
    raw_csv: Path | str = Path("Traffic_Collisions_280340447332117481.csv"),
    output_dir: Path | str = Path("artifacts"),
    cache_dir: Path | str = Path("cache"),
    target_zones: int = 50,
    refresh_cache: bool = False,
) -> dict:
    # The pipeline orchestrates the entire build from raw collisions through
    # generated map artifacts and the summary JSON consumed by the app.
    raw_csv = Path(raw_csv)
    output_dir = Path(output_dir)
    cache_dir = Path(cache_dir)
    ensure_directory(output_dir)
    ensure_directory(cache_dir)

    paths = BuildPaths.create(raw_csv=raw_csv, output_dir=output_dir, cache_dir=cache_dir)
    valid_collisions, invalid_collisions, raw_summary = clean_collision_data(paths.raw_csv)
    boundary = fetch_kw_boundary(paths.boundary_geojson, refresh_cache=refresh_cache)
    streets = fetch_kw_streets(boundary, paths.streets_geojson, refresh_cache=refresh_cache)
    blocks, boundary_proj = build_street_blocks(boundary, streets)
    zones, block_assignments, graph = build_analysis_zones(blocks, target_zones=target_zones)
    cleaned_collisions, metrics, join_summary = assign_collisions_to_zones(valid_collisions, zones)
    validations = validate_outputs(
        raw_summary=raw_summary,
        cleaned_collisions=cleaned_collisions,
        blocks=blocks,
        zones=zones,
        metrics=metrics,
        assignments=block_assignments,
        graph=graph,
        boundary_proj=boundary_proj,
    )

    zones_out = zones.merge(metrics, on=["zone_id", "zone_area_km2"], how="left")
    write_geojson(zones_out, paths.zones_geojson)
    cleaned_collisions.to_csv(paths.cleaned_csv, index=False)
    invalid_collisions.to_csv(paths.unmappable_csv, index=False)
    metrics.to_csv(paths.metrics_csv, index=False)

    summary = {
        "raw_summary": raw_summary,
        "join_summary": join_summary,
        "zone_count": int(len(zones)),
        "zone_area_km2_min": float(zones["zone_area_km2"].min()),
        "zone_area_km2_median": float(zones["zone_area_km2"].median()),
        "zone_area_km2_max": float(zones["zone_area_km2"].max()),
        "output_files": {
            "collisions_cleaned_csv": str(paths.cleaned_csv),
            "collisions_unmappable_csv": str(paths.unmappable_csv),
            "zones_geojson": str(paths.zones_geojson),
            "zone_metrics_csv": str(paths.metrics_csv),
            "build_summary_json": str(paths.summary_json),
        },
        "validations": validations,
    }
    with paths.summary_json.open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)
    return summary


def build_parser() -> argparse.ArgumentParser:
    # Command-line entrypoint for rebuilding the cached geography and exported
    # heatmap datasets from the raw collision CSV.
    parser = argparse.ArgumentParser(description="Build the KW collision heatmap artifacts.")
    parser.add_argument("--raw-csv", default="Traffic_Collisions_280340447332117481.csv", help="Path to the raw collision CSV.")
    parser.add_argument("--output-dir", default="artifacts", help="Directory for generated artifacts.")
    parser.add_argument("--cache-dir", default="cache", help="Directory for cached GIS inputs.")
    parser.add_argument("--target-zones", default=50, type=int, help="Target number of balanced street-bounded zones.")
    parser.add_argument("--refresh-cache", action="store_true", help="Refetch municipal boundary and OSM street data.")
    return parser


def main() -> None:
    args = build_parser().parse_args()
    summary = run_pipeline(
        raw_csv=args.raw_csv,
        output_dir=args.output_dir,
        cache_dir=args.cache_dir,
        target_zones=args.target_zones,
        refresh_cache=args.refresh_cache,
    )
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


## Build Entry Point

Source: `build_kw_heatmap.py`


In [ ]:
from kw_heatmap.pipeline import main


if __name__ == "__main__":
    # Thin wrapper so the repo can rebuild artifacts with
    # `python build_kw_heatmap.py`.
    main()


## Negative Sampling

Source: `negative_sampling.py`


In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
# INPUT: The cleaned CSV output by pipeline.py (post zone assignment).
#        If that file doesn't exist yet, set USE_RAW_FALLBACK = True and it
#        will run directly from the original raw CSV instead.
CLEANED_CSV   = "artifacts/collisions_cleaned.csv"
RAW_CSV       = "Traffic_Collisions_280340447332117481.csv"
OUTPUT_CSV    = "artifacts/collisions_with_negatives.csv"

# Ratio of negative samples to positive samples (1.0 = balanced 1:1)
NEGATIVE_RATIO = 1.0
RANDOM_STATE   = 42

# ---------------------------------------------------------------------------
# Categorical columns that data_processing.py OHE-encodes.
# Negatives need plausible string values for these — all other columns can
# be left as NaN since data_processing.py drops them anyway.
# "Other" is excluded because data_processing.py drops any row containing it.
# ---------------------------------------------------------------------------
CAT_COLS = [
    "ACCIDENT_WEEKDAY",
    "ACCIDENTLOCATION",
    "LIGHT",
    "ROADJURISDICTION",
    "TRAFFICCONTROL",
    "TRAFFICCONTROLCONDITION",
    "ENVIRONMENTCONDITION1",
]


def _empirical_values(series: pd.Series) -> tuple[list, list]:
    """
    Return (values, probabilities) from a Series, excluding 'Other'/'other'.
    Probabilities are normalised so they sum to 1.
    """
    counts = series.value_counts()
    counts = counts[~(counts.index.str.lower() == "other")]
    probs = counts / counts.sum()
    return probs.index.tolist(), probs.values.tolist()


def generate_negative_samples(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """
    Build a DataFrame of synthetic non-collision rows in the same raw
    column format as df (pre-OHE categorical strings).

    Allocation strategy:
      - If zone_id is present: proportional per zone (busier zones get more
        negatives, reflecting higher traffic exposure).
      - If zone_id is absent: global sampling from the full dataset.
    """
    n_pos = len(df)
    n_neg = int(round(n_pos * NEGATIVE_RATIO))
    has_zones = "zone_id" in df.columns

    date_min = pd.to_datetime(df["ACCIDENTDATE"]).min()
    date_max = pd.to_datetime(df["ACCIDENTDATE"]).max()
    date_range_s = int((date_max - date_min).total_seconds())

    # Pre-compute empirical distributions for categorical cols (global)
    global_dists = {
        col: _empirical_values(df[col].dropna()) for col in CAT_COLS
    }

    def _make_block(source_df: pd.DataFrame, n: int,
                    zone_id=None, zone_area=None) -> pd.DataFrame:
        """Generate n negative rows drawn from source_df's distributions."""
        block = pd.DataFrame(index=range(n))

        # Random timestamps within dataset date range
        offsets = rng.integers(0, date_range_s + 1, size=n)
        block["ACCIDENTDATE"] = [
            date_min + timedelta(seconds=int(s)) for s in offsets
        ]

        # Categorical features sampled from empirical distribution
        for col in CAT_COLS:
            vals, probs = _empirical_values(source_df[col].dropna())
            if not vals:          # fallback to global if zone has no data
                vals, probs = global_dists[col]
            block[col] = rng.choice(vals, size=n, p=probs)

        # Street notes — randomly draw from real notes in the source pool
        notes = source_df["XMLIMPORTNOTES"].dropna().values
        if len(notes):
            block["XMLIMPORTNOTES"] = rng.choice(notes, size=n)

        # Zone metadata (only when zone_id is present)
        if zone_id is not None:
            block["zone_id"]       = zone_id
            block["zone_area_km2"] = zone_area

        return block

    # Proportional per-zone allocation (preferred)
    if has_zones:
        print("  zone_id found — using proportional per-zone negative sampling.")
        zone_counts    = df["zone_id"].value_counts()
        zone_neg_share = ((zone_counts / n_pos) * n_neg).round().astype(int)

        # Fix any rounding drift
        diff = n_neg - zone_neg_share.sum()
        if diff:
            zone_neg_share[zone_neg_share.idxmax()] += diff

        blocks = []
        for zone_id, n_zone_neg in zone_neg_share.items():
            if n_zone_neg <= 0:
                continue
            zone_df   = df[df["zone_id"] == zone_id]
            zone_area = float(zone_df["zone_area_km2"].iloc[0])
            blocks.append(_make_block(zone_df, n_zone_neg, zone_id, zone_area))
        negatives = pd.concat(blocks, ignore_index=True)

    # ------------------------------------------------------------------
    # Global fallback (zone map not yet merged into CSV)
    # ------------------------------------------------------------------
    else:
        print("  zone_id not found — using global negative sampling.")
        print("  Re-run after pipeline.py has been run to get per-zone sampling.")
        negatives = _make_block(df, n_neg)

    return negatives


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import os

    # ---- Load source data ----
    if os.path.exists(CLEANED_CSV):
        print(f"Loading {CLEANED_CSV} ...")
        df = pd.read_csv(CLEANED_CSV)
        print("  (Using pipeline-cleaned CSV with zone assignments)")
    else:
        print(f"{CLEANED_CSV} not found — falling back to raw CSV.")
        print(f"Loading {RAW_CSV} ...")
        df = pd.read_csv(RAW_CSV)
        print("  (Zone-proportional sampling unavailable until pipeline.py is run)")

    print(f"  Rows loaded: {len(df)}")

    # ---- Add CRASH column to positives ----
    df["CRASH"] = 1

    # ---- Generate negatives ----
    rng = np.random.default_rng(RANDOM_STATE)
    print(f"\nGenerating {int(round(len(df) * NEGATIVE_RATIO))} negative samples ...")
    negatives = generate_negative_samples(df, rng)
    negatives["CRASH"] = 0

    # ---- Combine ----
    # Align columns so both DataFrames match, then shuffle
    combined = pd.concat([df, negatives], ignore_index=True)
    combined = combined.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    # ---- Save ----
    os.makedirs("artifacts", exist_ok=True)
    combined.to_csv(OUTPUT_CSV, index=False)

    n_pos = (combined["CRASH"] == 1).sum()
    n_neg = (combined["CRASH"] == 0).sum()
    print(f"\nDone.")
    print(f"  Collision rows  : {n_pos}")
    print(f"  No-collision rows: {n_neg}")
    print(f"  Total rows       : {len(combined)}")
    print(f"\nSaved to: {OUTPUT_CSV}")
    print("\nNext step: point data_processing.py at this file instead of")
    print("collisions_cleaned.csv, and make sure the CRASH column is not")
    print("in its columns_to_del list so it passes through the OHE step.")

## Data Processing

Source: `data_processing.py`


In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("artifacts/collisions_with_negatives.csv")

columns_to_del = ["OBJECTID", "ACCIDENTNUM", "ACCIDENT_YEAR", "ACCIDENT_MONTH", "ACCIDENT_DAY", "ACCIDENT_HOUR", "ACCIDENT_MINUTE", "ACCIDENT_SECOND", "XCOORD", "YCOORD", "LONGITUDE", "LATITUDE", "COLLISIONTYPE", "CLASSIFICATIONOFACCIDENT", "IMPACTLOCATION", "INITIALDIRECTIONOFTRAVELONE", "INITIALDIRECTIONOFTRAVELTWO", "INITIALIMPACTTYPE", "INTTRAFFICCONTROL", "LIGHTFORREPORT", "THRULANENO", "NORTHBOUNDDISOBEYCOUNT", "SOUTHBOUNDDISOBEYCOUNT", "PEDESTRIANINVOLVED", "CYCLISTINVOLVED", "MOTORCYCLISTINVOLVED", "ENVIRONMENTCONDITION2", "SELFREPORTED", "LASTEDITEDDATE", "CREATE_BY", "CREATE_DATE", "x", "y", "source_row", "year", "has_valid_coords", "distance_to_zone_m", "XMLIMPORTNOTES"]

df_updated = df.drop(columns=columns_to_del)
df_updated = df_updated[~df_updated.astype(str).apply(lambda x: x.str.lower()).isin(['other']).any(axis=1)]
df_updated['TRAFFICCONTROLCONDITION'] = df_updated['TRAFFICCONTROLCONDITION'].replace('unknown', 'Not Applicable')
df_updated['ACCIDENTDATE'] = pd.to_datetime(df_updated['ACCIDENTDATE'], format='mixed', errors='coerce')
df_updated['zone_id'] = df_updated['zone_id'].astype('category')
df_updated['zone_area_km2'] = df_updated['zone_area_km2'].astype('float64')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['ACCIDENT_WEEKDAY']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['ACCIDENT_WEEKDAY']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='ACCIDENT_WEEKDAY')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['ACCIDENTLOCATION']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['ACCIDENTLOCATION']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='ACCIDENTLOCATION')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['LIGHT']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['LIGHT']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='LIGHT')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['ROADJURISDICTION']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['ROADJURISDICTION']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='ROADJURISDICTION')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['TRAFFICCONTROL']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['TRAFFICCONTROL']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='TRAFFICCONTROL')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['TRAFFICCONTROLCONDITION']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['TRAFFICCONTROLCONDITION']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='TRAFFICCONTROLCONDITION')

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_updated[['ENVIRONMENTCONDITION1']])
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['ENVIRONMENTCONDITION1']), index=df_updated.index)
df_updated = pd.concat([df_updated, df_encoded], axis=1)
df_updated = df_updated.drop(columns='ENVIRONMENTCONDITION1')

df_updated.to_csv("Traffic_Collisions_Updated.csv")

## Model Training

Source: `train_model.py`


In [ ]:
import json
import os
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split

RANDOM_STATE = 42
DATA_FILE = "Traffic_Collisions_Updated.csv"
ARTIFACTS_DIR = "artifacts"


def load_data(path: str):
    print(f"Loading data from {path}...")
    df = pd.read_csv(path)
    print(f"  {len(df)} rows, {df.shape[1]} columns")

    if "ACCIDENTDATE" in df.columns:
        df["ACCIDENTDATE"] = pd.to_datetime(df["ACCIDENTDATE"], errors="coerce").map(
            lambda d: d.toordinal() if pd.notna(d) else 0
        )

    if "zone_id" in df.columns:
        df = pd.get_dummies(df, columns=["zone_id"], prefix="zone")

    drop_cols = ["CRASH", "Unnamed: 0"]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["CRASH"]

    print(f"  Features: {X.shape[1]}  |  Target distribution: {y.value_counts().to_dict()}")
    return X, y


def main():
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)

    X, y = load_data(DATA_FILE)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\nTrain: {len(X_train)} rows  |  Test: {len(X_test)} rows")

    # ------------------------------------------------------------------ #
    # GRADIENT BOOSTING — hyperparameter tuning via RandomizedSearchCV     #
    # ------------------------------------------------------------------ #
    print("\nTuning GradientBoosting (20 iterations, 3-fold CV)...")
    search = RandomizedSearchCV(
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        param_distributions={
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "max_depth": [3, 4, 5, 6],
            "subsample": [0.6, 0.8, 1.0],
            "min_samples_leaf": [1, 2, 4],
        },
        n_iter=20,
        scoring="roc_auc",
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    model = search.best_estimator_
    print(f"  Best params: {search.best_params_}")

    # ------------------------------------------------------------------ #
    # EVALUATION                                                           #
    # ------------------------------------------------------------------ #
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)

    print(f"\n  Accuracy            : {acc:.4f}")
    print(f"  ROC-AUC             : {auc:.4f}")
    print(f"  CV ROC-AUC (5-fold) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Crash", "Crash"]))
    print("  Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color="steelblue", lw=2, label=f"GradientBoosting (AUC = {auc:.4f})")
    plt.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Gradient Boosting")
    plt.legend(loc="lower right")
    plt.tight_layout()
    roc_path = os.path.join(ARTIFACTS_DIR, "roc_curve.png")
    plt.savefig(roc_path, dpi=150)
    plt.show()
    print(f"  ROC curve saved -> {roc_path}")

    # ------------------------------------------------------------------ #
    # SAVE                                                                 #
    # ------------------------------------------------------------------ #
    joblib.dump(model, os.path.join(ARTIFACTS_DIR, "collision_model.pkl"))
    joblib.dump(list(X.columns), os.path.join(ARTIFACTS_DIR, "model_features.pkl"))

    summary = {
        "model_type": "GradientBoosting",
        "best_params": search.best_params_,
        "roc_auc": round(auc, 6),
        "accuracy": round(acc, 6),
        "cv_mean": round(cv_scores.mean(), 6),
        "cv_std": round(cv_scores.std(), 6),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_features": X.shape[1],
        "trained_at": datetime.now(timezone.utc).isoformat(),
    }
    with open(os.path.join(ARTIFACTS_DIR, "model_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\nSaved model    -> artifacts/collision_model.pkl")
    print(f"Saved features -> artifacts/model_features.pkl")
    print(f"Saved summary  -> artifacts/model_summary.json")


if __name__ == "__main__":
    main()


## Model Training for Multiple Models

Source: `train_multiple_models.py`

**IMPORTANT

This code was used to see which model was best fit for our project. It wasn't used for the final model we chose. The final model is in `train_model.py`

In [ ]:
import json
import os
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_FILE = "Traffic_Collisions_Updated.csv"
ARTIFACTS_DIR = "artifacts"


def load_data(path: str):
    print(f"Loading data from {path}...")
    df = pd.read_csv(path)
    print(f"  {len(df)} rows, {df.shape[1]} columns")

    drop_cols = ["CRASH", "ACCIDENTDATE", "zone_id", "Unnamed: 0"]
    # Convert ACCIDENTDATE to a numeric ordinal so the model can use it
    if "ACCIDENTDATE" in df.columns:
        df["ACCIDENTDATE"] = pd.to_datetime(df["ACCIDENTDATE"], errors="coerce").map(
            lambda d: d.toordinal() if pd.notna(d) else 0
        )

    # One-hot encode zone_id so the model can use zone identity as a feature
    if "zone_id" in df.columns:
        df = pd.get_dummies(df, columns=["zone_id"], prefix="zone")

    drop_cols = ["CRASH", "Unnamed: 0"]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["CRASH"]

    print(f"  Features: {X.shape[1]}  |  Target distribution: {y.value_counts().to_dict()}")
    return X, y


def evaluate(name: str, model, X_test, y_test, X_train=None, y_train=None):
    print(f"\n--- {name} ---")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"  Accuracy : {acc:.4f}")
    print(f"  ROC-AUC  : {auc:.4f}")

    cv_mean, cv_std = None, None
    if X_train is not None and y_train is not None:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
        cv_mean, cv_std = cv_scores.mean(), cv_scores.std()
        print(f"  CV ROC-AUC (5-fold): {cv_mean:.4f} ± {cv_std:.4f}")

    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Crash", "Crash"]))
    print("  Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return acc, auc, cv_mean, cv_std


def tune_model(name: str, model, param_dist: dict, X_train, y_train, X_test, y_test, n_iter=20):
    print(f"\nTuning {name} ({n_iter} iterations, 3-fold CV)...")
    search = RandomizedSearchCV(
        model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="roc_auc",
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    best = search.best_estimator_

    print(f"  Best params: {search.best_params_}")
    y_prob = best.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, best.predict(X_test))
    print(f"  Tuned Accuracy : {acc:.4f}")
    print(f"  Tuned ROC-AUC  : {auc:.4f}")

    return best, acc, auc, search.best_params_


def print_feature_importance(model, feature_names, top_n=15):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    top = importances.sort_values(ascending=False).head(top_n)
    print(f"\nTop {top_n} Feature Importances (Random Forest):")
    for feat, score in top.items():
        print(f"  {feat:<45} {score:.4f}")


def main():
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)

    X, y = load_data(DATA_FILE)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\nTrain: {len(X_train)} rows  |  Test: {len(X_test)} rows")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    all_results = {}

    # ------------------------------------------------------------------ #
    # 1. BASELINE MODELS                                                   #
    # ------------------------------------------------------------------ #
    print("\n" + "=" * 60)
    print("BASELINE MODELS")
    print("=" * 60)

    # Logistic Regression — tune C and l1_ratio (0=l2, 1=l1)
    lr_base = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver="saga")
    lr_best, lr_acc, lr_auc, lr_params = tune_model(
        "LogisticRegression", lr_base,
        {
            "C": [0.001, 0.01, 0.1, 1, 10, 100],
            "l1_ratio": [0.0, 0.5, 1.0],
        },
        X_train_scaled, y_train, X_test_scaled, y_test,
        n_iter=12,
    )
    cv_scores = cross_val_score(lr_best, X_train_scaled, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    print(f"  CV ROC-AUC (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    all_results["LogisticRegression"] = {
        "accuracy": lr_acc, "roc_auc": lr_auc,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
        "best_params": lr_params,
    }

    # Decision Tree — tune depth and criterion
    dt_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
    dt_best, dt_acc, dt_auc, dt_params = tune_model(
        "DecisionTree", dt_base,
        {
            "max_depth": [3, 5, 7, 10, None],
            "criterion": ["gini", "entropy"],
            "min_samples_leaf": [1, 2, 4],
        },
        X_train, y_train, X_test, y_test,
        n_iter=15,
    )
    cv_scores = cross_val_score(dt_best, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    print(f"  CV ROC-AUC (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    all_results["DecisionTree"] = {
        "accuracy": dt_acc, "roc_auc": dt_auc,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
        "best_params": dt_params,
    }

    # ------------------------------------------------------------------ #
    # 2. ADVANCED MODELS                                                   #
    # ------------------------------------------------------------------ #
    print("\n" + "=" * 60)
    print("ADVANCED MODELS")
    print("=" * 60)

    # Random Forest — tune
    rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    rf_best, rf_acc, rf_auc, rf_params = tune_model(
        "RandomForest", rf_base,
        {
            "n_estimators": [100, 200, 300, 500],
            "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2", 0.5],
        },
        X_train, y_train, X_test, y_test,
    )
    cv_scores = cross_val_score(rf_best, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    print(f"  CV ROC-AUC (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    all_results["RandomForest"] = {
        "accuracy": rf_acc, "roc_auc": rf_auc,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
        "best_params": rf_params,
    }

    # Gradient Boosting — tune
    gb_base = GradientBoostingClassifier(random_state=RANDOM_STATE)
    gb_best, gb_acc, gb_auc, gb_params = tune_model(
        "GradientBoosting", gb_base,
        {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "max_depth": [3, 4, 5, 6],
            "subsample": [0.6, 0.8, 1.0],
            "min_samples_leaf": [1, 2, 4],
        },
        X_train, y_train, X_test, y_test,
    )
    cv_scores = cross_val_score(gb_best, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    print(f"  CV ROC-AUC (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    all_results["GradientBoosting"] = {
        "accuracy": gb_acc, "roc_auc": gb_auc,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
        "best_params": gb_params,
    }

    # Hist Gradient Boosting (sklearn XGBoost equivalent)
    print("\nTraining HistGradientBoosting...")
    hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE
    )
    hgb.fit(X_train, y_train)
    hgb_acc, hgb_auc, hgb_cv_mean, hgb_cv_std = evaluate(
        "HistGradientBoosting", hgb, X_test, y_test, X_train, y_train
    )
    all_results["HistGradientBoosting"] = {
        "accuracy": hgb_acc, "roc_auc": hgb_auc,
        "cv_mean": hgb_cv_mean, "cv_std": hgb_cv_std,
        "best_params": {"max_iter": 300, "learning_rate": 0.05, "max_depth": 4},
    }

    print_feature_importance(rf_best, list(X.columns))

    # ------------------------------------------------------------------ #
    # 3. SUMMARY COMPARISON                                                #
    # ------------------------------------------------------------------ #
    print("\n" + "=" * 60)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 60)
    print(f"\n  {'Model':<28} {'Accuracy':>10} {'ROC-AUC':>10} {'CV Mean':>10} {'CV Std':>8}")
    print(f"  {'-'*28} {'-'*10} {'-'*10} {'-'*10} {'-'*8}")
    for name, m in all_results.items():
        cv_m = f"{m['cv_mean']:.4f}" if m["cv_mean"] else "   N/A"
        cv_s = f"{m['cv_std']:.4f}" if m["cv_std"] else "   N/A"
        print(f"  {name:<28} {m['accuracy']:>10.4f} {m['roc_auc']:>10.4f} {cv_m:>10} {cv_s:>8}")

    # ------------------------------------------------------------------ #
    # 4. SAVE BEST OVERALL MODEL                                           #
    # ------------------------------------------------------------------ #
    trained_map = {
        "LogisticRegression": lr_best,
        "DecisionTree": dt_best,
        "RandomForest": rf_best,
        "GradientBoosting": gb_best,
        "HistGradientBoosting": hgb,
    }

    best_name = max(all_results, key=lambda n: all_results[n]["roc_auc"])
    best_model = trained_map[best_name]
    best_metrics = all_results[best_name]
    print(f"\nBest model: {best_name} (ROC-AUC = {best_metrics['roc_auc']:.4f})")

    model_path = os.path.join(ARTIFACTS_DIR, "collision_model.pkl")
    features_path = os.path.join(ARTIFACTS_DIR, "model_features.pkl")
    summary_path = os.path.join(ARTIFACTS_DIR, "model_summary.json")

    joblib.dump(best_model, model_path)
    joblib.dump(list(X.columns), features_path)

    summary = {
        "model_type": best_name,
        "roc_auc": round(best_metrics["roc_auc"], 6),
        "accuracy": round(best_metrics["accuracy"], 6),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_features": X.shape[1],
        "all_results": {
            k: {
                "accuracy": round(v["accuracy"], 6),
                "roc_auc": round(v["roc_auc"], 6),
                "cv_mean": round(v["cv_mean"], 6) if v["cv_mean"] else None,
                "cv_std": round(v["cv_std"], 6) if v["cv_std"] else None,
                "best_params": v["best_params"],
            }
            for k, v in all_results.items()
        },
        "trained_at": datetime.now(timezone.utc).isoformat(),
    }
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\nSaved model      -> {model_path}")
    print(f"Saved features   -> {features_path}")
    print(f"Saved summary    -> {summary_path}")


if __name__ == "__main__":
    main()


## Streamlit App

Source: `app.py`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pydeck as pdk
import streamlit as st

# Output locations produced by the build pipeline.
ARTIFACT_DIR = Path("artifacts")
SUMMARY_PATH = ARTIFACT_DIR / "build_summary.json"
ZONES_PATH = ARTIFACT_DIR / "zones.geojson"
COLLISIONS_PATH = ARTIFACT_DIR / "collisions_cleaned.csv"
COLOR_STOPS = [
    (255, 255, 204),
    (254, 217, 118),
    (253, 141, 60),
    (240, 59, 32),
    (177, 0, 38),
]


def interpolate_color(value: float, max_value: float) -> list[int]:
    if max_value <= 0:
        return [240, 240, 240, 160]
    ratio = min(max(value / max_value, 0.0), 1.0)
    scaled = ratio * (len(COLOR_STOPS) - 1)
    lower_index = int(scaled)
    upper_index = min(lower_index + 1, len(COLOR_STOPS) - 1)
    blend = scaled - lower_index
    lower = COLOR_STOPS[lower_index]
    upper = COLOR_STOPS[upper_index]
    rgb = [
        int(lower[channel] + (upper[channel] - lower[channel]) * blend)
        for channel in range(3)
    ]
    return [*rgb, int(150 + 80 * ratio)]


# Cached artifact loaders keep the app responsive by reusing the generated files
# instead of re-reading them on every UI interaction.
@st.cache_data(show_spinner=False)
def load_summary() -> dict:
    with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
        return json.load(handle)


@st.cache_data(show_spinner=False)
def load_zones() -> gpd.GeoDataFrame:
    with ZONES_PATH.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    return gpd.GeoDataFrame.from_features(payload["features"], crs="EPSG:4326")


@st.cache_data(show_spinner=False)
def load_collisions() -> pd.DataFrame:
    collisions = pd.read_csv(COLLISIONS_PATH, parse_dates=["ACCIDENTDATE"])
    collisions["year"] = collisions["year"].astype(int)
    return collisions


def prepare_map_data(
    zones: gpd.GeoDataFrame,
    collisions: pd.DataFrame,
    year_range: tuple[int, int],
    metric: str,
) -> tuple[dict, pd.DataFrame]:
    # Recompute zone-level metrics from the filtered collisions so the map and
    # table stay in sync with the selected year range and display metric.
    filtered = collisions[
        (collisions["year"] >= year_range[0]) & (collisions["year"] <= year_range[1])
    ].copy()
    year_span = max(year_range[1] - year_range[0] + 1, 1)

    grouped = filtered.groupby("zone_id").size().rename("collision_count").reset_index()
    display = zones[["zone_id", "zone_area_km2", "geometry"]].copy()
    display = display.merge(grouped, on="zone_id", how="left")
    display["collision_count"] = display["collision_count"].fillna(0).astype(int)
    display["collisions_per_km2"] = display["collision_count"] / display["zone_area_km2"]
    display["avg_collisions_per_year"] = display["collision_count"] / year_span

    metric_column = "collision_count" if metric == "Collision count" else "collisions_per_km2"
    max_value = float(display[metric_column].max())
    display["fill_color"] = display[metric_column].apply(
        lambda value: interpolate_color(float(value), max_value)
    )
    display["zone_area_km2"] = display["zone_area_km2"].round(3)
    display["collisions_per_km2"] = display["collisions_per_km2"].round(2)
    display["avg_collisions_per_year"] = display["avg_collisions_per_year"].round(2)

    payload = json.loads(display.to_json(drop_id=True))
    return payload, display.drop(columns=["geometry"])


def build_map(zones_geojson: dict) -> pdk.Deck:
    # Center the map on the generated zone geometry rather than hard-coding a
    # viewport, so regenerated zones still open in the right place.
    temp_gdf = gpd.GeoDataFrame.from_features(zones_geojson["features"], crs="EPSG:4326")
    centroid = temp_gdf.to_crs("EPSG:3347").unary_union.centroid
    center = gpd.GeoSeries([centroid], crs="EPSG:3347").to_crs("EPSG:4326").iloc[0]

    layer = pdk.Layer(
        "GeoJsonLayer",
        data=zones_geojson,
        pickable=True,
        stroked=True,
        filled=True,
        auto_highlight=True,
        get_fill_color="properties.fill_color",
        get_line_color=[60, 60, 60, 180],
        line_width_min_pixels=1,
    )
    return pdk.Deck(
        layers=[layer],
        initial_view_state=pdk.ViewState(
            latitude=center.y,
            longitude=center.x,
            zoom=10.4,
            pitch=0,
        ),
        tooltip={
            "html": (
                "<b>{zone_id}</b><br/>"
                "Area: {zone_area_km2} km²<br/>"
                "Collisions: {collision_count}<br/>"
                "Collisions / km²: {collisions_per_km2}<br/>"
                "Avg / year: {avg_collisions_per_year}"
            )
        },
        map_style="light",
    )


def main() -> None:
    # The app is intentionally read-only: it expects the build script to have
    # already generated the artifacts it visualizes.
    st.set_page_config(page_title="KW Collision Heatmap", layout="wide")
    st.title("Kitchener-Waterloo Collision Zone Heatmap")

    if not SUMMARY_PATH.exists() or not ZONES_PATH.exists() or not COLLISIONS_PATH.exists():
        st.error("Heatmap artifacts are missing. Run `python build_kw_heatmap.py` first.")
        st.stop()

    summary = load_summary()
    zones = load_zones()
    collisions = load_collisions()
    year_min = int(collisions["year"].min())
    year_max = int(collisions["year"].max())

    with st.sidebar:
        st.header("Controls")
        year_range = st.slider(
            "Year range",
            min_value=year_min,
            max_value=year_max,
            value=(year_min, year_max),
        )
        metric = st.radio(
            "Map metric",
            options=["Collision count", "Collisions per km²"],
            index=0,
        )
        st.metric("Excluded zero-coordinate rows", summary["raw_summary"]["unmappable_rows"])
        st.caption("These records remain in the artifacts but are excluded from the map.")
        warnings = summary["validations"].get("geometry_constrained_area_band_warnings", [])
        if warnings:
            st.caption(
                "Geometry-constrained zones outside the target area band: "
                + ", ".join(warnings)
            )

    zones_geojson, table = prepare_map_data(zones, collisions, year_range, metric)
    filtered_collisions = collisions[
        (collisions["year"] >= year_range[0]) & (collisions["year"] <= year_range[1])
    ]

    stat_1, stat_2, stat_3 = st.columns(3)
    stat_1.metric("Visible collisions", f"{len(filtered_collisions):,}")
    stat_2.metric("Zone count", f"{len(zones):,}")
    stat_3.metric("Year span", f"{year_range[0]}-{year_range[1]}")

    st.pydeck_chart(build_map(zones_geojson), use_container_width=True)
    st.subheader("Zone Metrics")
    st.dataframe(
        table.sort_values(by="collision_count", ascending=False).reset_index(drop=True),
        use_container_width=True,
        hide_index=True,
    )


if __name__ == "__main__":
    main()
